In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
import pickle
import os
import warnings

# Check for PyTensor compiler availability
try:
    from pytensor import config
    if not config.cxx:
        print("Warning: PyTensor is using Python backend due to missing C++ compiler. Performance may be degraded. Install g++ or set PYTENSOR_FLAGS=cxx=.")
except ImportError:
    print("Warning: PyTensor import failed. Ensure pymc3 and dependencies are installed correctly.")

def load_data(price_file, event_file):
    """Load and preprocess Brent oil price and event data."""
    # Load price data
    try:
        data = pd.read_csv(price_file)
        data['Date'] = pd.to_datetime(data['Date'], format='%d-%b-%y')
        data = data.sort_values('Date')
        data = data[(data['Date'] >= '2012-01-01') & (data['Date'] <= '2022-09-30')]
        if data.empty:
            raise ValueError("No data found in the specified date range (2012–2022).")
    except FileNotFoundError:
        raise FileNotFoundError(f"Price file {price_file} not found. Ensure it exists in data/raw/.")
    
    # Load event data
    try:
        events = pd.read_csv(event_file)
        events['Event_Date'] = pd.to_datetime(events['Event_Date'], format='%Y-%m-%d')
    except FileNotFoundError:
        raise FileNotFoundError(f"Event file {event_file} not found. Ensure it exists in data/processed/.")
    
    return data, events

def plot_eda(data, output_dir):
    """Perform EDA and save price and log return plots."""
    data['Log_Price'] = np.log(data['Price'])
    data['Log_Return'] = data['Log_Price'].diff()
    
    plt.style.use('ggplot')  # Use built-in Matplotlib style
    plt.figure(figsize=(12, 6))
    
    plt.subplot(2, 1, 1)
    plt.plot(data['Date'], data['Price'], label='Brent Oil Price (USD)')
    plt.title('Brent Oil Prices (2012–2022)')
    plt.ylabel('Price (USD)')
    plt.legend()
    
    plt.subplot(2, 1, 2)
    plt.plot(data['Date'], data['Log_Return'], label='Log Returns', color='orange')
    plt.title('Log Returns of Brent Oil Prices')
    plt.ylabel('Log Return')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'C:/Users/IE/Desktop/Week 10/eda_plots.png'))
    plt.close()

def build_change_point_model(prices, t):
    """Build and run Bayesian Change Point model."""
    with pm.Model() as model:
        # Set random seed for reproducibility
        np.random.seed(42)
        pm.set_seed(42)
        
        # Prior for switch point
        tau = pm.DiscreteUniform('tau', lower=0, upper=len(prices)-1)
        
        # Priors for mean prices
        mu_1 = pm.Normal('mu_1', mu=np.mean(prices), sd=10)
        mu_2 = pm.Normal('mu_2', mu=np.mean(prices), sd=10)
        
        # Prior for standard deviation
        sigma = pm.HalfNormal('sigma', sd=10)
        
        # Switch function
        mu = pm.math.switch(tau >= t, mu_1, mu_2)
        
        # Likelihood
        pm.Normal('likelihood', mu=mu, sd=sigma, observed=prices)
        
        # MCMC sampling
        trace = pm.sample(2000, tune=1000, return_inferencedata=True)
    
    return trace

def save_model_output(trace, output_dir):
    """Save MCMC trace for reproducibility."""
    with open(os.path.join(output_dir, 'C:/Users/IE/Desktop/Week 10/change_point_trace.pkl'), 'wb') as f:
        pickle.dump(trace, f)

def analyze_results(trace, data, events, output_dir):
    """Analyze model output, associate with events, and visualize results."""
    # Check convergence
    summary = az.summary(trace, var_names=['tau', 'mu_1', 'mu_2', 'sigma'])
    print("Model Summary:")
    print(summary)
    
    # Plot trace for diagnostics
    az.plot_trace(trace, var_names=['tau', 'mu_1', 'mu_2'])
    plt.savefig(os.path.join(output_dir, 'C:/Users/IE/Desktop/Week 10/trace_plots.png'))
    plt.close()
    
    # Identify change point
    tau_posterior = trace.posterior['tau'].values.flatten()
    tau_mode = int(np.bincount(tau_posterior).argmax())
    change_date = data['Date'].iloc[tau_mode]
    print(f"\nMost likely change point: {change_date.date()}")
    
    # Quantify impact
    mu_1_mean = trace.posterior['mu_1'].mean().values
    mu_2_mean = trace.posterior['mu_2'].mean().values
    price_change = ((mu_2_mean - mu_1_mean) / mu_1_mean) * 100
    print(f"Mean price before: ${mu_1_mean:.2f}, after: ${mu_2_mean:.2f}, change: {price_change:.2f}%")
    
    # Associate with events (30-day window)
    tolerance = pd.Timedelta(days=30)
    matched_events = events[(events['Event_Date'] >= change_date - tolerance) & 
                           (events['Event_Date'] <= change_date + tolerance)]
    
    print("\nMatched Events:")
    if matched_events.empty:
        print("No events found within 30 days of the change point.")
    else:
        for _, event in matched_events.iterrows():
            print(f"{event['Event_Date'].date()}: {event['Event_Description']} ({event['Event_Type']})")
    
    # Plot prices with change point and events
    plt.style.use('ggplot')  # Use built-in Matplotlib style
    plt.figure(figsize=(12, 6))
    plt.plot(data['Date'], data['Price'], label='Brent Oil Price (USD)')
    plt.axvline(x=change_date, color='red', linestyle='--', label=f'Change Point: {change_date.date()}')
    for idx, event in events.iterrows():
        plt.axvline(x=event['Event_Date'], color='green', alpha=0.3, linestyle=':', 
                    label=f"Event: {event['Event_Description']} ({event['Event_Date'].date()})" if idx == 0 else "")
    plt.title('Brent Oil Prices with Change Point and Events')
    plt.xlabel('Date')
    plt.ylabel('Price (USD)')
    plt.legend()
    plt.savefig(os.path.join(output_dir, 'C:/Users/IE/Desktop/Week 10/price_with_change_point.png'))
    plt.close()

def main():
    """Main function to execute Task 2 analysis."""
    # Define paths
    price_file = 'C:/Users/IE/Desktop/Week 10/data/raw/BrentOilPrices.csv'
    event_file = 'C:/Users/IE/Desktop/Week 10/data/processed/events.csv'
    output_dir = 'C:/Users/IE/Desktop/Week 10/outputs/figures'
    model_dir = 'C:/Users/IE/Desktop/Week 10/outputs/models'
    
    # Create output directories if they don't exist
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)
    
    # Load data
    try:
        data, events = load_data(price_file, event_file)
    except Exception as e:
        print(f"Error loading data: {e}")
        return
    
    # Perform EDA
    plot_eda(data, output_dir)
    
    # Build and run model
    prices = data['Price'].values
    t = np.arange(len(prices))
    try:
        trace = build_change_point_model(prices, t)
    except Exception as e:
        print(f"Error running model: {e}")
        return
    
    # Save model output
    save_model_output(trace, model_dir)
    
    # Analyze results
    analyze_results(trace, data, events, output_dir)

if __name__ == '__main__':
    main()